# MI–RFE–SVM CICFLOW — local progress, Drive checkpoint 5 giờ

Giữ mục đích từng cell theo notebook ColabPro: mount → source → dependencies/tests → dataset → contract → preflight → artifacts → local cache → run/resume → progress/checkpoint → selection → refit → Test một lần → verify. Mặc định **preflight**. Chỉ Cell 9/13/14 được bật rõ ràng mới khởi chạy worker. PROVISIONAL_RESEARCH / OFFICIAL_BLOCKED. Không có kết quả chạy sẵn trong notebook.


### Cell 1 — Mount Drive
Dataset đã có trên Drive; không tải hoặc copy Parquet.


In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
DRIVE = Path('/content/drive/MyDrive')
assert DRIVE.is_dir()
print('DRIVE:', DRIVE)


### Cell 2 — Chốt source/commit và nạp adapter
Điền full commit SHA sau khi bạn tự push. Checkout cũ chỉ được dùng nếu sạch và đúng SHA; không pull/reset.


In [ ]:
if globals().get('JOB', {}).get('running'):
    raise RuntimeError('Job đang chạy; không đổi source/commit trong lúc này')
import os, sys, json, re, subprocess, threading, time, uuid
from datetime import datetime, timezone
APPROVED_COMMIT = ''  # 40 ký tự, lấy bằng git rev-parse HEAD sau push
URL = 'https://github.com/kasum00/RFforNetwork.git'
if not re.fullmatch(r'[0-9a-fA-F]{40}', APPROVED_COMMIT):
    raise ValueError('Điền APPROVED_COMMIT đủ 40 ký tự')
APPROVED_COMMIT = APPROVED_COMMIT.lower()
REPO = Path('/content/CICFLOW_SVM5H_' + APPROVED_COMMIT[:12])
if not REPO.exists():
    subprocess.run(['git', 'clone', '--', URL, str(REPO)], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', APPROVED_COMMIT], check=True)
def git(*args):
    return subprocess.run(['git', '-C', str(REPO), *args], text=True, capture_output=True, check=True).stdout.strip()
assert git('rev-parse', 'HEAD') == APPROVED_COMMIT
assert not git('status', '--porcelain'), 'Checkout có thay đổi: dùng source sạch, không reset'
SOURCE_ROOT = REPO / 'code_CICFLOW'
sys.dont_write_bytecode = True
sys.path.insert(0, str(SOURCE_ROOT)) if str(SOURCE_ROOT) not in sys.path else None
from mi_rfe_svm_colab.common import CONFIGS, core, contract, paths, read, redact
import mi_rfe_svm_colab.common as loaded_common
if Path(loaded_common.__file__).resolve() != (SOURCE_ROOT/'mi_rfe_svm_colab/common.py').resolve():
    raise RuntimeError('Kernel đang giữ source cũ; restart session rồi dùng đúng commit')
print('SOURCE:', SOURCE_ROOT, 'COMMIT:', APPROVED_COMMIT)


### Cell 3 — Dependencies và synthetic tests
Không tự cài package. Nếu cần, bật INSTALL_DEPENDENCIES rồi restart session và chạy lại Cell 1–3. Tests dùng mock SVC, không fit SVC thật.


In [ ]:
INSTALL_DEPENDENCIES = False
if INSTALL_DEPENDENCIES:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r',
                    str(SOURCE_ROOT / 'mi_rfe_svm_colab/requirements-colab.txt')], check=True)
    raise RuntimeError('Đã cài theo yêu cầu. Restart session rồi chạy lại Cell 1–3, đặt INSTALL_DEPENDENCIES=False.')
RUN_SYNTHETIC_TESTS = True
EVIDENCE = {'status': 'NOT_RUN'}
UNIT_TEST_ARTIFACTS = {}
if RUN_SYNTHETIC_TESTS:
    evidence_path = Path('/content') / ('svm5h_tests_' + uuid.uuid4().hex + '.json')
    command = [sys.executable, '-B', '-m', 'mi_rfe_svm_colab.selftest',
               '--source', str(SOURCE_ROOT), '--evidence', str(evidence_path)]
    env = os.environ.copy(); env['PYTHONDONTWRITEBYTECODE'] = '1'
    from mi_rfe_svm_colab.common import info
    test_start = datetime.now(timezone.utc).isoformat()
    test_log = evidence_path.with_suffix('.log')
    test_process = subprocess.Popen(command, cwd=str(SOURCE_ROOT), env=env,
                                    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    with test_log.open('x') as output:
        for line in test_process.stdout:
            line = redact(line); output.write(line); output.flush(); print(line, end='', flush=True)
    test_exit = test_process.wait()
    test_record = evidence_path.with_name(evidence_path.stem + '_process.json')
    test_record.write_text(json.dumps({'start': test_start, 'end': datetime.now(timezone.utc).isoformat(),
                                      'command': command, 'native_exit_code': test_exit}, indent=2))
    UNIT_TEST_ARTIFACTS = {p.name: {'path': str(p), **info(p)} for p in
                           (evidence_path, test_log, test_record) if p.exists()}
    EVIDENCE = read(evidence_path) if evidence_path.exists() else {'status': 'FAIL'}
    if test_exit or EVIDENCE.get('status') != 'PASS':
        raise RuntimeError('Synthetic tests chưa PASS. Không khởi chạy full.')
print('TEST EVIDENCE:', EVIDENCE)


### Cell 4 — Paths và cấu hình
Chọn một hoặc nhiều cấu hình; queue chạy tuần tự. Giữ run01 như yêu cầu; nếu run cũ khác identity, chọn run02 cho run mới. Không ghi đè run cũ.


In [ ]:
DATASET_ROOT = DRIVE / 'CICFLOWMETER_FEATURE_CSV'
OUTPUT_ROOT = DRIVE / 'experiment_results'
LOCAL_ROOT = Path('/content/mi_rfe_svm_5h')
SELECTED_CONFIGS = ['FW256_FS8']  # đổi thành list(CONFIGS) nếu muốn queue cả bốn
RUN_TAG = 'run01'
assert SELECTED_CONFIGS and len(set(SELECTED_CONFIGS)) == len(SELECTED_CONFIGS)
assert set(SELECTED_CONFIGS) <= set(CONFIGS)
for config in CONFIGS:
    assert (DATASET_ROOT / config).is_dir(), config
    print(config, '->', OUTPUT_ROOT / f'mi_rfe_svm_{config}_{RUN_TAG}')


### Cell 5 — Scientific contract
Một MI trên 380 feature Train; RF-RFE SVM depth15; SVC RBF với cache10000, C10, max_iter20000. Không đổi cache để qua resource gate.


In [ ]:
print(json.dumps(contract(SOURCE_ROOT), indent=2))
print('Drive checkpoint interval: 18000 seconds; no mid-fit SVC checkpoint')


### Cell 6 — Resource preflight
Đây là budget vận hành, không phải thay siêu tham số. Gate BLOCKED thì dừng; không giảm dữ liệu/k/cây. Sau khi đổi mode/reuse, worker kiểm tra lại mọi binding.


In [ ]:
BUDGET = {'min_available_ram_bytes': 2*1024**3,
          'max_process_rss_bytes': 40*1024**3}
REUSE_FROM = {config: None for config in CONFIGS}
# Có thể điền thư mục reference-layout MI/RFE-SVM cũ có identity và đủ state/hash.
# Không trỏ tới Stage4 per-feature MI, RF ranking hay XGBoost.
TAKEOVER_CONFIRMATIONS = {config: None for config in CONFIGS}
def request_for(config, mode='preflight', phase=None):
    request = {'source': str(SOURCE_ROOT), 'dataset': str(DATASET_ROOT),
               'output': str(OUTPUT_ROOT), 'local_root': str(LOCAL_ROOT),
               'config': config, 'run_tag': RUN_TAG, 'mode': mode,
               'git_commit': APPROVED_COMMIT, 'budget': BUDGET,
               'unit_tests': EVIDENCE, 'unit_test_artifacts': UNIT_TEST_ARTIFACTS,
               'reuse_from': REUSE_FROM.get(config),
               'takeover_confirmation': TAKEOVER_CONFIRMATIONS.get(config),
               'acknowledge_local_stop': globals().get('ACKNOWLEDGE_LOCAL_STOP', False)}
    if phase: request['phase'] = phase
    return request
SESSION_LOGS = Path('/content/svm5h_notebook_sessions')
SESSION_LOGS.mkdir(exist_ok=True)
def launch(request):
    command = [sys.executable, '-B', '-u', '-m', 'mi_rfe_svm_colab.runner']
    stamp = uuid.uuid4().hex
    started = datetime.now(timezone.utc).isoformat()
    env = os.environ.copy(); env['PYTHONDONTWRITEBYTECODE'] = '1'
    process = subprocess.Popen(command, cwd=str(request['source']), env=env, stdin=subprocess.PIPE,
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                               bufsize=1, errors='replace')
    process.stdin.write(json.dumps(request)); process.stdin.close()
    result = None
    try:
        with (SESSION_LOGS / (stamp + '.log')).open('x') as log:
            for line in process.stdout:
                line = redact(line); log.write(line); log.flush()
                if line.startswith('__SVM5H_RESULT__'):
                    if result is not None: raise RuntimeError('Duplicate worker result')
                    result = json.loads(line[len('__SVM5H_RESULT__'):])
                else:
                    print(line, end='', flush=True)
    except BaseException as exc:
        local, _ = paths(request)
        try:
            local.mkdir(parents=True, exist_ok=True)
            (local/'STOP_REQUESTED').write_text('Parent requested graceful stop after output/logging error')
        except OSError as flag_error:
            print('LOCAL STOP FLAG NOT SAVED:', flag_error, flush=True)
        print('Waiting for child safe boundary; it may still be fitting. Do not start a duplicate job.', flush=True)
        for remaining in process.stdout:
            print(redact(remaining), end='', flush=True)
        process.wait()
        raise RuntimeError('Parent output failed; child exited with ' + str(process.returncode)) from exc
    native_exit = process.wait()
    (SESSION_LOGS / (stamp + '.json')).write_text(json.dumps({
        'start': started, 'end': datetime.now(timezone.utc).isoformat(), 'command': command,
        'config': request['config'], 'mode': request['mode'], 'phase': request.get('phase'),
        'native_exit_code': native_exit, 'reported_status': (result or {}).get('status')}, indent=2))
    print(request['config'], 'EXIT:', native_exit, 'STATUS:', (result or {}).get('status'))
    if result is None:
        raise RuntimeError(f'Worker exit {native_exit}, missing result; inspect local session log')
    if native_exit != 0 or result.get('status') == 'BLOCKED':
        print('REASONS:', result.get('reasons', []))
        raise RuntimeError(f'Worker BLOCKED/FAILED, native exit={native_exit}')
    return result
PREFLIGHT = {}
for config in SELECTED_CONFIGS:
    try:
        PREFLIGHT[config] = launch(request_for(config))
        print(json.dumps({'config': config, 'gate': PREFLIGHT[config].get('resource_gate', 'SAVED_COMPLETED_RUN_VERIFIED')}, indent=2))
    except Exception as exc:
        PREFLIGHT[config] = {'status': 'BLOCKED', 'error': str(exc)}
        print(config, 'BLOCKED:', exc)


### Cell 7 — Xác minh artifact MI/RFE và checkpoint cũ
Chỉ đọc identity/metadata/hash; không fit. Nếu dùng REUSE_FROM, điền ở Cell 6 rồi chạy lại Cell 6–7. Không nhập checkpoint CSV cũ hoặc MI theo feature vào biến thể MI full380.


In [ ]:
from mi_rfe_svm_colab.preflight import review
for config in SELECTED_CONFIGS:
    plan = review(request_for(config), need_fit=False)
    print(config, 'IMPORT:', plan['identity']['reuse'])
    print('LATEST COMMITTED:', plan['latest_committed_checkpoint'])
    print('UNRELEASED LEASE:', plan['unreleased_lease'])
    print('REASONS:', plan['reasons'])
# Nếu runtime trước đã mất: xác nhận nó thực sự dừng rồi điền chính xác:
# TAKEOVER_CONFIRMATIONS['FW256_FS8'] = 'PREVIOUS RUNTIME STOPPED LEASE_<id in ở trên>'


### Cell 8 — Local caching và mode
Worker đọc Train/Validation tối đa một lần mỗi split trong một attempt, lưu cache npy và scaler có checksum trong /content. K sau đó dùng lại mảng đã scale/rank. Không copy Parquet và không upload cache.


In [ ]:
MODE = 'preflight'  # preflight | run | resume | verify | finalize
ACKNOWLEDGE_LOCAL_STOP = False  # True khi chủ động resume sau graceful stop của chính run này
for config in SELECTED_CONFIGS:
    local, remote = paths(request_for(config))
    print(config, 'LOCAL:', local, 'DRIVE:', remote)
print('MODE:', MODE)


### Cell 9 — K-search full/resume hoặc preflight/verify
Đặt MODE=run cho run mới, resume cho run đang dở. Queue tuần tự trong background để Cell 10–11 vẫn dùng được. Cell trả về chỉ có nghĩa đã khởi động; xem trạng thái JOB. Cell này không final refit hay Test.


In [ ]:
def start_queue(requests):
    global JOB, JOB_THREAD
    requests = json.loads(json.dumps(requests))  # freeze queue requests
    if globals().get('JOB', {}).get('running'):
        raise RuntimeError('Worker/queue đang chạy; không tạo job trùng')
    JOB = {'running': True, 'current': None, 'results': [], 'error': None, 'stop_queue': False}
    def work():
        try:
            for request in requests:
                if JOB['stop_queue']: break
                JOB['current'] = request['config']
                result = launch(request)
                JOB['results'].append({'config': request['config'], **result})
                if result.get('status') == 'PAUSED_AT_SAFE_CHECKPOINT': break
        except BaseException as exc:
            JOB['error'] = str(exc)
            print('QUEUE STOPPED:', exc, flush=True)
        finally:
            JOB['running'] = False
            JOB['current'] = None
            print('QUEUE FINISHED; review JOB and saved checkpoint status. No automatic retry.', flush=True)
    JOB_THREAD = threading.Thread(target=work, daemon=True)
    JOB_THREAD.start()
if MODE in ('run', 'resume'):
    requests = []
    for config in SELECTED_CONFIGS:
        req = request_for(config, MODE)
        name = paths(req)[1].name
        exact = f'{MODE.upper()} PROVISIONAL {name}'
        if input('Nhập đúng: ' + exact + '\n') != exact: raise RuntimeError('Confirmation differs')
        req['confirmation'] = exact; requests.append(req)
    start_queue(requests)
elif MODE in ('preflight', 'verify'):
    for config in SELECTED_CONFIGS:
        result = launch(request_for(config, MODE))
        print(json.dumps({k: result[k] for k in ('status','best_k','directory','metrics') if k in result}, indent=2))
else:
    print('MODE=finalize: dùng Cell 13 rồi Cell 14 với xác nhận riêng.')


### Cell 10 — Xem local progress
Đọc file local; không tạo heartbeat hoặc file mới trên Drive. Chạy lại cell này khi cần xem tiến độ.


In [ ]:
print('JOB:', {k: v for k, v in globals().get('JOB', {}).items() if k != 'results'})
for config in SELECTED_CONFIGS:
    local, remote = paths(request_for(config))
    for p in (local/'state/progress.json', local/'sync_receipt.json'):
        print(config, p.name, read(p) if p.exists() else 'NOT_PRESENT')
print('SVC.fit không hỗ trợ checkpoint giữa chừng; chạy lâu không đồng nghĩa bị treo.')


### Cell 11 — Yêu cầu checkpoint/dừng an toàn
ACTION=checkpoint hoặc stop chỉ tạo cờ local; worker xử lý sau k/phase hoàn thành. Không upload giữa SVC.fit. Không dùng Disconnect để mong lưu phần fit đang dở.


In [ ]:
ACTION = 'none'  # none | checkpoint | stop
if ACTION != 'none':
    if ACTION not in ('checkpoint', 'stop'): raise ValueError(ACTION)
    if not globals().get('JOB', {}).get('running') or not JOB.get('current'):
        raise RuntimeError('Không có worker đang chạy trong notebook này')
    config = JOB['current']
    local, remote = paths(request_for(config))
    flag = local / ('STOP_REQUESTED' if ACTION == 'stop' else 'CHECKPOINT_REQUESTED')
    local.mkdir(parents=True, exist_ok=True)
    flag.write_text(datetime.now(timezone.utc).isoformat())
    if ACTION == 'stop': JOB['stop_queue'] = True
    print('REQUESTED LOCALLY:', flag, '; wait for safe-boundary COMMITTED and PAUSED status')
else:
    print('Không gửi yêu cầu. Checkpoint tự động sau >=18000s tại safe boundary.')


### Cell 12 — Best-k selection đã khóa
Chỉ hiển thị sau khi đủ k=1..100. Hòa Accuracy chọn k nhỏ nhất. Không đọc Test.


In [ ]:
if globals().get('JOB', {}).get('running'): raise RuntimeError('Chờ queue hoàn thành/dừng an toàn')
for config in SELECTED_CONFIGS:
    local, remote = paths(request_for(config))
    p = local / 'state/selection_frozen.json'
    print(config, read(p) if p.exists() else 'SELECTION_NOT_YET_FROZEN')


### Cell 13 — Final refit Train+Validation
Bật riêng sau khi selection đã khóa. Tạo scaler mới và SVC mới như reference; không refit MI/RFE. Hoàn thành sẽ lưu model/scaler vào bundle Drive. Cell này không đọc Test.


In [ ]:
CONFIRM_FINAL_REFIT = False
if CONFIRM_FINAL_REFIT:
    requests = []
    for config in SELECTED_CONFIGS:
        req = request_for(config, 'finalize', 'final_refit')
        exact = 'FINALIZE PROVISIONAL ' + paths(req)[1].name
        if input('Nhập đúng: ' + exact + '\n') != exact: raise RuntimeError('Confirmation differs')
        req['confirmation'] = exact; requests.append(req)
    start_queue(requests)
else:
    print('Final refit chưa được bật. Không fit/predict.')


### Cell 14 — One-time Test evaluation
Chỉ sau final refit đã COMMITTED. Claim ghi/readback trên Drive trước lần mở Test đầu tiên. Mất runtime sau claim nhưng chưa có kết quả hợp lệ sẽ BLOCKED, không tự đánh giá lại.


In [ ]:
CONFIRM_TEST_ONCE = False
if CONFIRM_TEST_ONCE:
    requests = []
    for config in SELECTED_CONFIGS:
        req = request_for(config, 'finalize', 'test')
        exact = 'FINALIZE PROVISIONAL ' + paths(req)[1].name
        if input('Nhập đúng: ' + exact + '\n') != exact: raise RuntimeError('Confirmation differs')
        req['confirmation'] = exact; requests.append(req)
    start_queue(requests)
else:
    print('Test chưa được bật. Không mở Test hoặc predict.')


### Cell 15 — Verify và đọc kết quả cuối
Chỉ hash/JSON đã lưu; không load model hoặc đọc/predict Test. PASS kỹ thuật không thay trạng thái khoa học.


In [ ]:
if globals().get('JOB', {}).get('running'): raise RuntimeError('Worker vẫn chạy, chưa thể gọi là hoàn thành')
summary = []
for config in SELECTED_CONFIGS:
    local, remote = paths(request_for(config))
    if not ((remote / 'RUN_COMPLETE.json').exists() or (remote / 'COMPLETE.json').exists()):
        summary.append({'config': config, 'status': 'INCOMPLETE', 'directory': str(remote)})
        continue
    result = launch(request_for(config, 'verify'))
    summary.append({'config': config, 'status': result['status'], 'best_k': result['best_k'],
                    'result_status': result['result_status'], **result['metrics'], 'directory': str(remote)})
print(json.dumps(summary, indent=2))
print('PROVISIONAL_RESEARCH / OFFICIAL_BLOCKED. Không phải official reproduction.')
